# Chicago City Data — SQL & Python EDA

**Author:** José Santos
**Tools:** Python · SQLite · SQL Magic · Pandas  
**Dataset:** City of Chicago Open Data Portal  

---

## Project Overview

This project explores three public datasets from the City of Chicago using SQL and Python.  
The datasets are loaded into a local SQLite database and queried to extract socioeconomic,  
educational, and crime-related insights.

**Datasets used:**
-  Census socioeconomic indicators by community area (2008–2012) [source](https://data.cityofchicago.org/Health-Human-Services/Census-Data-Selected-socioeconomic-indicators-in-C/kn9c-c2s2/about_data)
-  Chicago public school profiles including safety and enrollment data [source](https://data.cityofchicago.org/Education/Chicago-Public-Schools-Progress-Report-Cards-2011-/9xs2-f89t/about_data)
-  Reported crimes in Chicago [source](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/about_data)


## Setup

### Install & Import Libraries


In [2]:
#!pip install pandas
#!pip install ipython-sql prettytable

import prettytable
prettytable.DEFAULT = 'DEFAULT'


### Connect to SQLite Database


In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('FinalDB.db')
cur = conn.cursor()


In [4]:
%load_ext sql


### Load CSV Files into SQLite

Place the three CSV files in the same directory as this notebook, then run the cell below.


In [5]:
df_economic = pd.read_csv("ChicagoCensusData.csv")
df_crime    = pd.read_csv("ChicagoCrimeData.csv")
df_schools  = pd.read_csv("ChicagoPublicSchools.csv")

df_economic.to_sql("SOCIO_ECONOMIC_FACTORS", conn, if_exists='replace', index=False)
df_crime.to_sql("CRIME_DATA",               conn, if_exists='replace', index=False)
df_schools.to_sql("PUBLIC_SCHOOLS",          conn, if_exists='replace', index=False)

print(f"Census rows    : {len(df_economic)}")
print(f"Crime rows     : {len(df_crime)}")
print(f"Schools rows   : {len(df_schools)}")


Census rows    : 78
Crime rows     : 533
Schools rows   : 566


In [6]:
%sql sqlite:///FinalDB.db


---

## Data Overview


In [ ]:
df_economic.info()


In [ ]:
df_crime.info()


In [16]:
df_schools.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566 entries, 0 to 565
Data columns (total 78 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   School_ID                                         566 non-null    int64  
 1   NAME_OF_SCHOOL                                    566 non-null    object 
 2   Elementary, Middle, or High School                566 non-null    object 
 3   Street_Address                                    566 non-null    object 
 4   City                                              566 non-null    object 
 5   State                                             566 non-null    object 
 6   ZIP_Code                                          566 non-null    int64  
 7   Phone_Number                                      566 non-null    object 
 8   Link                                              565 non-null    object 
 9   Network_Manager      

In [15]:
%%sql
SELECT * FROM SOCIO_ECONOMIC_FACTORS LIMIT 5;


 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX
1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0


---

## Analysis


### Question 1 — Total Number of Crimes
How many crimes are recorded in the dataset?


In [14]:
%%sql
SELECT COUNT(*) AS total_crimes FROM CRIME_DATA


 * sqlite:///FinalDB.db
Done.


total_crimes
533


### Question 2 — Low Income Community Areas
List community areas with a per capita income below 11,000.


In [7]:
%%sql
SELECT COMMUNITY_AREA_NAME, COMMUNITY_AREA_NUMBER
FROM SOCIO_ECONOMIC_FACTORS
WHERE PER_CAPITA_INCOME < 11000


 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,COMMUNITY_AREA_NUMBER
West Garfield Park,26.0
South Lawndale,30.0
Fuller Park,37.0
Riverdale,54.0


### Question 3 — Crimes Involving Minors
List all case numbers for crimes involving minors.
> Note: crimes categorised as involving *children* are excluded — only offences explicitly referencing *minors* are counted.


In [9]:
%%sql
SELECT CASE_NUMBER FROM CRIME_DATA
WHERE DESCRIPTION LIKE '%MINOR%'


 * sqlite:///FinalDB.db
Done.


CASE_NUMBER
HL266884
HK238408


### Question 4 — Kidnapping Crimes Involving a Child
List all kidnapping crimes where the victim is described as a child.


In [10]:
%%sql
SELECT *
FROM CRIME_DATA
WHERE PRIMARY_TYPE = 'KIDNAPPING' AND DESCRIPTION LIKE '%CHILD%'


 * sqlite:///FinalDB.db
Done.


ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,LATITUDE,LONGITUDE,LOCATION
5276766,HN144152,2007-01-26,050XX W VAN BUREN ST,1792,KIDNAPPING,CHILD ABDUCTION/STRANGER,STREET,0,0,1533,15,29.0,25.0,20,1143050.0,1897546.0,2007,41.87490841,-87.75024931,"(41.874908413, -87.750249307)"


### Question 5 — Crime Types at Schools
What distinct types of crimes have been recorded at school locations?


In [11]:
%%sql
SELECT DISTINCT PRIMARY_TYPE
FROM CRIME_DATA
WHERE LOCATION_DESCRIPTION LIKE '%SCHOOL%'


 * sqlite:///FinalDB.db
Done.


PRIMARY_TYPE
BATTERY
CRIMINAL DAMAGE
NARCOTICS
ASSAULT
CRIMINAL TRESPASS
PUBLIC PEACE VIOLATION


### Question 6 — Average Safety Score by School Type
What is the average safety score for each school type (Elementary, Middle, High)?


In [20]:
%%sql
SELECT `Elementary, Middle, or High School`, ROUND(AVG(SAFETY_SCORE),2) as avg_safety_Score
FROM PUBLIC_SCHOOLS
GROUP BY `Elementary, Middle, or High School`
ORDER BY avg_safety_Score DESC

 * sqlite:///FinalDB.db
Done.


"Elementary, Middle, or High School",avg_safety_Score
HS,49.62
ES,49.52
MS,48.0


### Question 7 — Highest Poverty Areas
Which 5 community areas have the highest percentage of households below the poverty line?


In [21]:
%%sql
SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY
FROM SOCIO_ECONOMIC_FACTORS
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC
LIMIT 5


 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4


### Question 8 — Most Crime-Prone Community Area
Which community area has the highest number of reported crimes? (Return area number only.)


In [22]:
%%sql
SELECT COMMUNITY_AREA_NUMBER
FROM SOCIO_ECONOMIC_FACTORS
WHERE COMMUNITY_AREA_NUMBER = 
                            (SELECT COMMUNITY_AREA_NUMBER
                            FROM CRIME_DATA 
                            GROUP BY COMMUNITY_AREA_NUMBER
                            ORDER BY count(*) DESC
                            LIMIT 1 )


 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER
25.0


### Question 9 — Highest Hardship Index (Subquery)
Find the name of the community area with the highest hardship index.


In [23]:
%%sql
SELECT COMMUNITY_AREA_NAME
FROM SOCIO_ECONOMIC_FACTORS
WHERE HARDSHIP_INDEX IN (SELECT MAX(HARDSHIP_INDEX)
                        FROM SOCIO_ECONOMIC_FACTORS )


 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Riverdale


### Question 10 — Most Crimes by Community Area Name (Subquery)
Find the name of the community area with the most reported crimes.


In [24]:
%%sql
SELECT COMMUNITY_AREA_NAME
FROM SOCIO_ECONOMIC_FACTORS
WHERE COMMUNITY_AREA_NUMBER = (
    SELECT COMMUNITY_AREA_NUMBER FROM CRIME_DATA
    GROUP BY COMMUNITY_AREA_NUMBER
    ORDER BY count(*) DESC LIMIT 1
    )

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Austin


---

## Wrap-Up


In [25]:
conn.close()
print("Connection closed.")


Connection closed.
